In [1]:
import numpy as np
import pandas as pd
import re

In [2]:
def parse_age_to_interval(age_str):
    # "0 to 4 years" → Interval(0, 4, closed='left')
    m = re.match(r'(\d+) to (\d+)', age_str)
    if m:
        return pd.Interval(int(m.group(1)), int(m.group(2)), closed='left')
    
    # "100+" → Interval(100, inf, closed='left')
    m = re.match(r'(\d+) years and ', age_str)
    if m:
        return pd.Interval(int(m.group(1)), np.inf, closed='left')
    
    return pd.NA

In [3]:
age_sex = pd.read_csv(
    '../data/bayesian_network/province-age-sex.csv',
    skiprows=[0,1,2,3,4,5,6,7,10,11],
    header=[0,1],
    index_col=0,
)[0:21]
age_sex.columns.names = ['geography', 'gender']
age_sex.index.name = 'age_group'

geo = (
    age_sex.columns.get_level_values('geography')
    .to_series()
    .replace(r'^Unnamed.*', pd.NA, regex=True)
    .ffill()
)
age_sex.columns = pd.MultiIndex.from_arrays(
    [geo.values, age_sex.columns.get_level_values('gender')],
    names=['geography', 'gender']
)

age_sex = age_sex.apply(lambda col: pd.to_numeric(col.astype(str).str.replace(',', ''), errors='coerce'))
age_sex.index = age_sex.index.map(parse_age_to_interval)

In [4]:
result = (
    age_sex
    .stack(['geography', 'gender'])
    .unstack('geography')
    .swaplevel('age_group', 'gender')
    .sort_index()
)
result['Canada (except provinces)'] = result.sum(axis=1)

In [5]:
result.loc['Men+']

geography,Alberta,British Columbia,Manitoba,New Brunswick,Newfoundland and Labrador,Nova Scotia,Ontario,Prince Edward Island,Quebec,Saskatchewan,Canada (except provinces)
age_group,,,,,,,,,,,
"[0.0, 4.0)",131507,114559,42204,17188,10041,21563,365599,3615,218495,36338,961109
"[5.0, 9.0)",146831,130524,46198,20019,11924,24838,402965,4369,240168,39998,1067834
"[10.0, 14.0)",147785,136018,45458,21147,13197,26219,419881,4583,244876,40476,1099640
"[15.0, 19.0)",137819,143876,44308,21076,14026,26067,443227,4678,225890,37127,1098094
"[20.0, 24.0)",142181,172054,56853,23635,15191,33807,539339,6550,247872,37913,1275395
"[25.0, 29.0)",156448,205321,54726,24116,14370,35632,601163,5845,286241,39440,1423302
"[30.0, 34.0)",175956,210865,51332,23779,14988,33721,570277,5001,296451,42060,1424430
"[35.0, 39.0)",183812,197455,49338,24432,15323,31819,520197,4849,286815,42895,1356935
"[40.0, 44.0)",169950,176719,45948,24694,15714,29804,471705,4807,303835,40357,1283533


In [7]:
result.to_csv('../data/bayesian_network/age-fixed.csv')

------------------------------------

In [8]:
data2 = pd.read_csv(
    '../data/bayesian_network/age-sex-health.csv',
    skiprows=[0,1,2,3,4,5,6,7,8,11,12],
    header=[0,1],
    index_col=0
)[:4]
data2.columns.names = ['age_group', 'sex']
data2.index.name = 'condition'

filled_columns = (
    data2.columns.get_level_values('age_group')
    .to_series()
    .replace(r'^Unnamed.*', pd.NA, regex=True)
    .ffill()
)

data2.columns = pd.MultiIndex.from_arrays(
    [filled_columns.values, data2.columns.get_level_values('sex')],
    names=['age_group', 'sex']
)

data2 = data2.apply(lambda col: col.astype(str).str.replace('E', ''))
data2 = data2.apply(lambda col: pd.to_numeric(col.astype(str).str.replace(',', ''), errors='coerce'))
data2 = data2.drop(columns=['Total, 18 years and over'])

C:\Users\ltyih\AppData\Local\Temp\ipykernel_24812\1872592946.py:24: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  data2 = data2.drop(columns=['Total, 18 years and over'])


In [9]:
data2.columns = pd.MultiIndex.from_arrays(
    [data2.columns.get_level_values('age_group').map(parse_age_to_interval),
     data2.columns.get_level_values('sex')],
    names=['age_group', 'sex']
)

In [10]:
data2

age_group                                          [18.0, 34.0)           \
sex                                                       Males  Females   
condition                                                                  
Body mass index, adjusted self-reported, adult ...       955200   974800   
High blood pressure 21 22                                122900    84600   
Current smoker, daily or occasional 23 24 25 26 27       562800   330200   
Influenza immunization in the past 12 months 28 29       647100  1037900   

age_group                                          [35.0, 49.0)           \
sex                                                       Males  Females   
condition                                                                  
Body mass index, adjusted self-reported, adult ...      1258100  1132100   
High blood pressure 21 22                                442000   308100   
Current smoker, daily or occasional 23 24 25 26 27       626900   425100   
Influenza immunization in the past 12 months 28 29       812500  1181800   

age_group                                          [50.0, 64.0)           \
sex                                                       Males  Females   
condition                                                                  
Body mass index, adjusted self-reported, adult ...      1284600  1185800   
High blood pressure 21 22                               1130600   913300   
Current smoker, daily or occasional 23 24 25 26 27       649900   572000   
Influenza immunization in the past 12 months 28 29      1215600  1454300   

age_group                                          [65.0, inf)           
sex                                                      Males  Females  
condition                                                                
Body mass index, adjusted self-reported, adult ...      869500  1045400  
High blood pressure 21 22                              1440800  1700900  
Current smoker, daily or occasional 23 24 25 26 27      325600   304300  
Influenza immunization in the past 12 months 28 29     1932500  2284600

In [11]:
pivot2 = (
    data2
    .stack(['age_group', 'sex'])
    .unstack(['condition'])
    .swaplevel('age_group', 'sex')
    .sort_index()
)

In [12]:
pivot2.to_csv('../data/bayesian_network/health-fixed.csv', index=True)